# Dependencies

In [67]:
import numpy as np
import pickle

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.linear_model import LogisticRegression


import sys
sys.path.append("/home/rguo_hpc/myfolder/mocap")
from datasets.augmentations import Augmentations
from swav.finetune.layers import ProjectionHead, PrototypeLayer
from swav.finetune.utils import sinkhorn, swav_loss
from swav.finetune.others.utils import PairedEmbeddingDataset, train_prototypes, compute_new_representations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [73]:
K = 128
sample_freq = 5

In [69]:
# Load skeletonMAE representations of original skeleton data
mae_tr  = np.load("./representations/sdannce_tr.npy")[:, ::sample_freq] # 360, 4500, 192
mae_val = np.load("./representations/sdannce_val.npy")[:, ::sample_freq]
mae_feats = np.concatenate([mae_tr, mae_val]) # 480, 4500/sample_freq, 192
num_seq, L, D = mae_feats.shape

# Load skeletonMAE representations of augmented skeleton data 
mae_tr_aug  = np.load("./representations/sdannce_tr.npy")[:, ::sample_freq]
mae_val_aug = np.load("./representations/sdannce_val.npy")[:, ::sample_freq]
mae_feats_aug = np.concatenate([mae_tr_aug, mae_val_aug]) # 480, 4500/sample_freq, 192

In [70]:
# Reshape
mae_feats = mae_feats.reshape(-1, D)
mae_feats_aug = mae_feats_aug.reshape(-1, D) # (240000, 192)
l_tr = int(mae_feats.shape[0] *3/4)

In [71]:
# load labels
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    data_fmr1 = pickle.load(file)
fmr1_fold_1 = {"train":[402, 404, 405, 406, 407, 408], "valid": [401, 403]} # In total 8 mice, each 3 sequnces with 90000 frames

hlac_labels = []
for mouse in fmr1_fold_1["train"]+fmr1_fold_1["valid"]:
    num_seq = len(data_fmr1[mouse]["hlac"])
    for i in range(num_seq):
        hlac = np.squeeze(data_fmr1[mouse]["hlac"][i])
        hlac_labels.append(hlac)

hlac_labels = np.array(hlac_labels).reshape(-1)[::sample_freq]
hlac_tr = hlac_labels[:l_tr]
hlac_val = hlac_labels[l_tr:]

In [101]:
NUM_PROTOTYPES = K         # <-- match your expected number of behaviors
NUM_EPOCHS = 20
BATCH_SIZE = 2048
PROJECTION_HIDDEN_DIM = 192
PROJECTION_OUT_DIM = 192
lr = 0.0001
weight_decay = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 192

prototypes, projection_head, dataset = train_prototypes(
                    mae_feats, mae_feats_aug, num_prototypes=NUM_PROTOTYPES, 
                    batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS, 
                    projection_hidden_dim = PROJECTION_HIDDEN_DIM, projection_out_dim=PROJECTION_OUT_DIM,
                    lr = lr, weight_decay=weight_decay) # gmm_means = centers
# save model
torch.save(prototypes.state_dict(), "trained_prototypes.pt")
if projection_head is not None:
    torch.save(projection_head.state_dict(), "trained_projection_head.pt")
    print("Saved trained_projection_head.pt")

epoch   0  step    0  loss 4.5949
epoch   0  step  100  loss 7.0965
epoch   0  step  200  loss 6.9237
epoch   0 done, avg loss 6.9889
epoch   1  step    0  loss 6.9220
epoch   1  step  100  loss 0.9439
epoch   1  step  200  loss 0.6730
epoch   1 done, avg loss 1.7519
epoch   2  step    0  loss 0.7441
epoch   2  step  100  loss 0.6499
epoch   2  step  200  loss 0.6484
epoch   2 done, avg loss 0.6582
epoch   3  step    0  loss 0.6092
epoch   3  step  100  loss 0.5702
epoch   3  step  200  loss 0.6117
epoch   3 done, avg loss 0.6253
epoch   4  step    0  loss 0.5757
epoch   4  step  100  loss 0.5960
epoch   4  step  200  loss 0.5756
epoch   4 done, avg loss 0.6003
epoch   5  step    0  loss 0.6071
epoch   5  step  100  loss 0.5738
epoch   5  step  200  loss 0.5689
epoch   5 done, avg loss 0.5788
epoch   6  step    0  loss 0.5721
epoch   6  step  100  loss 0.5537
epoch   6  step  200  loss 0.5656
epoch   6 done, avg loss 0.5646
epoch   7  step    0  loss 0.5616
epoch   7  step  100  loss 0

In [102]:
# After training
# Load prototypes model
prototypes =  PrototypeLayer(embed_dim=EMBED_DIM, num_prototypes=NUM_PROTOTYPES)
checkpoint_model = torch.load("trained_prototypes.pt", map_location=DEVICE, weights_only=False)
prototypes.load_state_dict(checkpoint_model, strict=True)

<All keys matched successfully>

In [103]:
which = "projection" if PROJECTION_OUT_DIM is not None else "cluster"
# Load projection head if exists
projection_head =  ProjectionHead(in_dim=EMBED_DIM, hidden_dim=PROJECTION_HIDDEN_DIM, out_dim=PROJECTION_OUT_DIM)
checkpoint = torch.load("trained_projection_head.pt", map_location=DEVICE, weights_only=False)
projection_head.load_state_dict(checkpoint, strict=True)

<All keys matched successfully>

In [104]:
new_repr = compute_new_representations(prototypes, torch.from_numpy(mae_feats).float(), DEVICE, 
                                       projection_head = projection_head, which="projection",
                                       #projection_head = None, which="cluster"
                                      )
mae_feats_tr = new_repr[:l_tr,]
mae_feats_val = new_repr[l_tr:,]

In [31]:
# Original MAE
model = LogisticRegression(max_iter=500, multi_class='multinomial')
model.fit(mae_tr.reshape(-1, D),  hlac_tr)

y_pred = model.predict(mae_val.reshape(-1, D))
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.6978611111111112

Classification Report:
               precision    recall  f1-score   support

           1       0.69      0.60      0.64     21208
           2       0.63      0.66      0.64     30498
           3       0.74      0.83      0.78      3936
           4       0.74      0.69      0.71      3893
           5       0.46      0.29      0.36      1701
           6       0.90      0.94      0.92     11090
           7       0.68      0.68      0.68     23037
           8       0.75      0.78      0.76     12637

    accuracy                           0.70    108000
   macro avg       0.70      0.68      0.69    108000
weighted avg       0.70      0.70      0.70    108000



/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [105]:
# After training
model = LogisticRegression(max_iter=500, multi_class='multinomial')
#model = RandomForestClassifier()
# fit & predict
model.fit(mae_feats_tr,  hlac_tr)

y_pred = model.predict(mae_feats_val)
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.6821944444444444

Classification Report:
               precision    recall  f1-score   support

           1       0.68      0.60      0.64     21208
           2       0.62      0.64      0.63     30498
           3       0.73      0.81      0.77      3936
           4       0.69      0.60      0.64      3893
           5       0.38      0.16      0.22      1701
           6       0.85      0.95      0.90     11090
           7       0.65      0.67      0.66     23037
           8       0.73      0.77      0.75     12637

    accuracy                           0.68    108000
   macro avg       0.67      0.65      0.65    108000
weighted avg       0.68      0.68      0.68    108000

